## PlanetMicrobe Sample Queries

In [31]:
spark = get_spark_session() ## acquire compute

<img src="img/planetmicrobe-ERD.jpg" alt="PlanetMicrobe ERD" height="1000" width="800">

#### Show all tables in the planetmicrobe_planetmicrobe namespace (planetmicrobe is the tenant name and planetmicrobe is the database/namespace name)

In [19]:
spark.sql("SHOW TABLES IN planetmicrobe_planetmicrobe").show(100,truncate=False)

+---------------------------+------------------------+-----------+
|namespace                  |tableName               |isTemporary|
+---------------------------+------------------------+-----------+
|planetmicrobe_planetmicrobe|spatial_ref_sys         |false      |
|planetmicrobe_planetmicrobe|app                     |false      |
|planetmicrobe_planetmicrobe|app_data_type           |false      |
|planetmicrobe_planetmicrobe|app_result              |false      |
|planetmicrobe_planetmicrobe|campaign                |false      |
|planetmicrobe_planetmicrobe|file_format             |false      |
|planetmicrobe_planetmicrobe|file_type               |false      |
|planetmicrobe_planetmicrobe|file                    |false      |
|planetmicrobe_planetmicrobe|go                      |false      |
|planetmicrobe_planetmicrobe|pfam                    |false      |
|planetmicrobe_planetmicrobe|project_type            |false      |
|planetmicrobe_planetmicrobe|project                 |false   

In [34]:
spark.sql("USE planetmicrobe_planetmicrobe")
spark.sql("DESCRIBE EXTENDED sample").show(truncate=False) ## Data Dictionary

+----------------------------+---------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                              |comment|
+----------------------------+---------------------------------------------------------------------------------------+-------+
|sample_id                   |string                                                                                 |NULL   |
|schema_id                   |string                                                                                 |NULL   |
|accn                        |string                                                                                 |NULL   |
|locations                   |string                                                                                 |NULL   |
|number_vals                 |array<int>                                                                       

### 1. Join Campaigns with Sampling Events and Count Events
This query combines the campaign and sampling_event tables, counting how many sampling events are associated with each campaign.

In [20]:
df = spark.sql("""
SELECT c.campaign_id, 
       c.name AS campaign_name, 
       COUNT(se.sampling_event_id) AS event_count
FROM planetmicrobe_planetmicrobe.campaign c
LEFT JOIN planetmicrobe_planetmicrobe.sampling_event se ON c.campaign_id = se.campaign_id
GROUP BY c.campaign_id, c.name
ORDER BY event_count DESC
""")
df.show()

+-----------+--------------+-----------+
|campaign_id| campaign_name|event_count|
+-----------+--------------+-----------+
|        106|TARA_20110928Z|         36|
|        114|TARA_20100624Z|         31|
|        107|TARA_20110826Z|         30|
|        129|TARA_20110726Z|         30|
|        105|TARA_20111124Z|         30|
|        118|TARA_20100309Z|         28|
|        126|TARA_20110401Z|         26|
|        112|TARA_20101231Z|         26|
|        122|TARA_20120223Z|         25|
|        109|TARA_20110520Z|         24|
|        111|TARA_20110311Z|         23|
|        108|TARA_20110626Z|         22|
|        113|TARA_20101126Z|         21|
|        127|TARA_20091111Z|         20|
|        133|TARA_20100905Z|         20|
|        128|TARA_20090926Z|         19|
|        124|TARA_20111230Z|         19|
|          1| AN10/KN197-08|         18|
|        132|TARA_20101003Z|         18|
|        116|TARA_20100410Z|         16|
+-----------+--------------+-----------+
only showing top

### 2. List Applications with Their Result Paths and Data Types
This query retrieves applications along with their result paths and the corresponding data types, showing all relationships.

In [21]:
df = spark.sql("""
SELECT a.app_id, 
       a.name AS app_name, 
       ar.path AS result_path, 
       adt.name AS data_type_name
FROM planetmicrobe_planetmicrobe.app a
JOIN planetmicrobe_planetmicrobe.app_result ar ON a.app_id = ar.app_id
JOIN planetmicrobe_planetmicrobe.app_data_type adt ON ar.app_data_type_id = adt.app_data_type_id
""")
df.show()

+------+--------------------+--------------------+--------------+
|app_id|            app_name|         result_path|data_type_name|
+------+--------------------+--------------------+--------------+
|     1|           libra-1.0|score/distance.ma...|        MATRIX|
|     2|  centrifuge-1.0.4u2|centrifuge-out/fi...|    CENTRIFUGE|
|     3| ohana-blast-0.1.2u1|         ohana-hits/|     BLAST-TAB|
|     4|mash-all-vs-all-0...|mash-out/figures/...|        MATRIX|
+------+--------------------+--------------------+--------------+



### 3. Average Total Bases for Runs Associated with Specific Projects¶
This query finds the average total bases for runs linked to a specified project.

In [22]:
df = spark.sql("""
SELECT p.project_id, 
       p.name AS project_name, 
       AVG(r.total_bases) AS avg_total_bases
FROM planetmicrobe_planetmicrobe.project p
JOIN planetmicrobe_planetmicrobe.run r ON p.project_id = r.experiment_id
GROUP BY p.project_id, p.name
ORDER BY avg_total_bases DESC
""")
df.show()

+----------+--------------------+---------------+
|project_id|        project_name|avg_total_bases|
+----------+--------------------+---------------+
|         7|CDEBI Juan de Fuc...|  2.925810428E9|
|         2|Amazon Plume Meta...|   2.10405514E9|
|         6|       BATS Chisholm|  1.991074826E9|
|         4|Amazon River Meta...|    1.6036242E9|
|         9|         HOT 224-283|  1.401753912E9|
|        11|         HOT 194-215|    1.2623163E9|
|        15|Tara Polar Circle...|  1.254251602E9|
|        10|         HOT 144-166|  1.123357252E9|
|         1|Amazon Plume Meta...|    1.0871259E9|
|         8|                 GOS|  1.043258424E9|
|         3|Amazon PolyA Meta...|   5.40133342E8|
|         5|Amazon River Meta...|   4.01667248E8|
|        12|HOT 194-215 Metat...|   4.00384956E8|
|        13|                 OSD|     3.965411E8|
|        14|         Tara Oceans|    8.4800996E7|
+----------+--------------------+---------------+



### 4. Taxa Richness per Sample
This query calculates the taxa richness for each sample, ensuring only samples with associated taxa are included.

In [24]:
df = spark.sql("""
SELECT s.sample_id, 
       COUNT(DISTINCT rt.tax_id) AS taxa_richness
FROM planetmicrobe_planetmicrobe.sample s
JOIN planetmicrobe_planetmicrobe.sample_to_sampling_event ste ON s.sample_id = ste.sample_id
JOIN planetmicrobe_planetmicrobe.run_to_taxonomy rt ON ste.sampling_event_id = rt.run_id
GROUP BY s.sample_id
ORDER BY taxa_richness DESC
""")
df.show()

+---------+-------------+
|sample_id|taxa_richness|
+---------+-------------+
|     2330|         4777|
|     1705|         4777|
|     1853|         4777|
|     1704|         4777|
|     1722|         4740|
|     2076|         4740|
|     1708|         4685|
|     2119|         4670|
|     2192|         4670|
|     2041|         4666|
|     2040|         4666|
|     2110|         4652|
|     2111|         4652|
|     2167|         4652|
|     2166|         4652|
|     2097|         4634|
|     2002|         4573|
|     1844|         4573|
|     2091|         4561|
|     2049|         4528|
+---------+-------------+
only showing top 20 rows


### 5. What are the predominant microbial taxa identified across different environmental sampling campaigns?

In [7]:
df = spark.sql("""
    SELECT rt.tax_id, 
           COUNT(rt.num_reads) AS total_reads, 
           se.campaign_id
    FROM planetmicrobe_planetmicrobe.run_to_taxonomy rt
    JOIN planetmicrobe_planetmicrobe.run r ON rt.run_id = r.run_id
    JOIN planetmicrobe_planetmicrobe.sampling_event se ON r.experiment_id = se.sampling_event_id
    GROUP BY rt.tax_id, 
             se.campaign_id
    ORDER BY total_reads DESC;
""")

df.show(10)

+------+-----------+-----------+
|tax_id|total_reads|campaign_id|
+------+-----------+-----------+
|155978|         36|       NULL|
|187493|         36|       NULL|
|  1280|         36|       NULL|
|   329|         36|       NULL|
|363852|         36|       NULL|
|198620|         36|       NULL|
|   691|         36|       NULL|
| 37372|         36|       NULL|
|367806|         36|       NULL|
|234827|         36|       NULL|
+------+-----------+-----------+
only showing top 10 rows


### 6. How do the count and types of files correlate with the completion times of runs in various sequencing experiments?

In [8]:
df = spark.sql("""
    SELECT r.run_id, 
           COUNT(rtf.file_id) AS file_count, 
           AVG(r.time_of_run) AS avg_run_time
    FROM planetmicrobe_planetmicrobe.run r
    JOIN planetmicrobe_planetmicrobe.run_to_file rtf ON r.run_id = rtf.run_id
    GROUP BY r.run_id;
""")
df.show()

+------+----------+------------+
|run_id|file_count|avg_run_time|
+------+----------+------------+
|   296|         1|        NULL|
|   467|         1|        NULL|
|   675|         2|        NULL|
|   691|         2|        NULL|
|   829|         2|        NULL|
|  1090|         1|        NULL|
|  1159|         2|        NULL|
|  1436|         2|        NULL|
|  1512|         2|        NULL|
|  1572|         1|        NULL|
|  2069|         1|        NULL|
|  2088|         2|        NULL|
|  2136|         2|        NULL|
|  2162|         2|        NULL|
|  2294|         2|        NULL|
|  2904|         1|        NULL|
|  3210|         2|        NULL|
|  3414|         2|        NULL|
|  3606|         2|        NULL|
|  3959|         2|        NULL|
+------+----------+------------+
only showing top 20 rows


### 7. Impact of Experimental Strategies on Library Quality:

In [9]:
df = spark.sql("""
    SELECT l.strategy, 
           AVG(l.length) AS avg_read_length, 
           AVG(r.total_spots) AS avg_total_spots
    FROM planetmicrobe_planetmicrobe.library l
    JOIN planetmicrobe_planetmicrobe.run r ON l.experiment_id = r.experiment_id
    GROUP BY l.strategy;
""")
df.show()

+--------+------------------+-------------------+
|strategy|   avg_read_length|    avg_total_spots|
+--------+------------------+-------------------+
|     WGA|171.82608695652175| 8.52139533465909E7|
| RNA-Seq|234.80695847362514|9.235747792878942E7|
|AMPLICON|195.22685185185185|   942423.764054165|
|   OTHER|              NULL|           439048.5|
|     EST|              NULL|  5772690.785714285|
|     WGS|  233.258922323303| 8.09329054340949E7|
+--------+------------------+-------------------+



### 8. Project Types and Data File Counts:

In [10]:
df = spark.sql("""
    SELECT pt.name AS project_type_name, 
           COUNT(pf.file_id) AS file_count
    FROM planetmicrobe_planetmicrobe.project_type pt
    JOIN planetmicrobe_planetmicrobe.project p ON pt.project_type_id = p.project_type_id
    JOIN planetmicrobe_planetmicrobe.project_to_file pf ON p.project_id = pf.project_id
    GROUP BY pt.name
    ORDER BY file_count DESC;
""")

df.show()

+--------------------+----------+
|   project_type_name|file_count|
+--------------------+----------+
|   marine metagenome|        13|
|marine metatransc...|         2|
+--------------------+----------+



In [30]:
spark.stop()